In [1]:
pip install pandas numpy scikit-learn matplotlib

In [3]:
# ============================================================
# FINAL STRESS-TEST TIMETABLE OPTIMIZER
# Includes: realistic class sizes, congestion, plots
# ============================================================

import pandas as pd
import numpy as np
import itertools
import matplotlib.pyplot as plt

# -----------------------------
# CONFIGURATION TOGGLE
# -----------------------------

STRESS_TEST = True   # <<< change to False for normal scenario

# -----------------------------
# LOAD DATA
# -----------------------------

tt = pd.read_csv("India_Analyzed_Realistic_OneWeek_Timetable.csv")
tt.columns = tt.columns.str.lower().str.strip()

# -----------------------------
# REALISTIC CLASS SIZES
# -----------------------------

if STRESS_TEST:
    tt["students"] = np.where(
        tt["room_type"].str.contains("lab", case=False, na=False),
        80,    # labs
        150    # theory
    )
else:
    tt["students"] = 50

# -----------------------------
# EXTRACT BLOCK & FLOOR
# -----------------------------

def extract_block_floor(room):
    try:
        p = room.split("-")
        return p[0], int(p[1][0])
    except:
        return None, None

tt["block"], tt["floor"] = zip(*tt["room_no"].astype(str).apply(extract_block_floor))

# -----------------------------
# COMPULSORY LAB LOCATIONS
# -----------------------------

COMPULSORY_BLOCK = {
    "vlsi lab": "E",
    "dsp lab": "E",
    "microprocessor lab": "E",
    "communication lab": "E",
    "chemistry lab": "CE"
}

tt["subject_lc"] = tt["subject"].str.lower()

def enforce_compulsory_block(row):
    if row["subject_lc"] in COMPULSORY_BLOCK:
        return COMPULSORY_BLOCK[row["subject_lc"]]
    return row["block"]

tt["block"] = tt.apply(enforce_compulsory_block, axis=1)

# -----------------------------
# BLOCK DISTANCES (METERS)
# -----------------------------

BLOCK_DISTANCE = {
    ("C", "E"): 180,
    ("C", "M"): 240,
    ("C", "CE"): 120,
    ("E", "M"): 200,
    ("E", "CE"): 90,
    ("M", "CE"): 160
}

def block_distance(b1, b2):
    if b1 == b2:
        return 0
    return BLOCK_DISTANCE.get((b1, b2)) or BLOCK_DISTANCE.get((b2, b1)) or 200

# -----------------------------
# LIFT SYSTEM PARAMETERS
# -----------------------------

TRANSITION_TIME = 10
LIFTS_PER_BLOCK = 2
LIFT_CAPACITY = 10
ROUND_TRIP_TIME = 2

TRIPS = TRANSITION_TIME // ROUND_TRIP_TIME
CAPACITY_PER_BLOCK = LIFTS_PER_BLOCK * TRIPS * LIFT_CAPACITY
# = 100 students

# -----------------------------
# METRICS
# -----------------------------

def compute_metrics(df):
    total_dist, total_students = 0, 0
    late_list = []

    df = df.sort_values(["day", "time"]).reset_index(drop=True)

    for i in range(len(df) - 1):
        a, b = df.iloc[i], df.iloc[i+1]
        if a["day"] != b["day"]:
            continue

        students = a["students"]
        total_students += students
        total_dist += block_distance(a["block"], b["block"]) * students

        if abs(a["floor"] - b["floor"]) > 2:
            demand = students
        else:
            demand = 0

        late = max(0, demand - CAPACITY_PER_BLOCK)
        late_list.append(late)

    avg_dist = total_dist / total_students if total_students else 0
    avg_late = np.mean(late_list) if late_list else 0
    max_late = max(late_list) if late_list else 0

    return avg_dist, avg_late, max_late, late_list

# -----------------------------
# BASELINE
# -----------------------------

base_dist, base_avg_late, base_max_late, base_late_series = compute_metrics(tt)

# -----------------------------
# OPTIMIZATION (SAFE)
# -----------------------------

optimized_days = []

for day, day_df in tt.groupby("day"):
    blocks = day_df["block"].unique().tolist()
    best_cost = float("inf")
    best_day = day_df

    for perm in itertools.permutations(blocks):
        candidate = pd.concat([day_df[day_df["block"] == b] for b in perm])
        avg_dist, avg_late, max_late, _ = compute_metrics(candidate)

        if max_late > base_max_late:
            continue

        cost = (avg_dist / base_dist + avg_late / (base_avg_late + 1)) / 2

        if cost < best_cost:
            best_cost = cost
            best_day = candidate

    optimized_days.append(best_day)

optimized = pd.concat(optimized_days).reset_index(drop=True)

opt_dist, opt_avg_late, opt_max_late, opt_late_series = compute_metrics(optimized)

# -----------------------------
# RESULTS
# -----------------------------

print("\n========== FINAL RESULTS ==========")
print("Stress Test Enabled:", STRESS_TEST)

print(f"\nBaseline Avg Distance : {base_dist:.2f} m")
print(f"Optimized Avg Distance: {opt_dist:.2f} m")

print(f"\nBaseline Avg Late Students : {base_avg_late:.2f}")
print(f"Optimized Avg Late Students: {opt_avg_late:.2f}")

print(f"\nBaseline Max Late Students : {base_max_late}")
print(f"Optimized Max Late Students: {opt_max_late}")

# -----------------------------
# PLOTS
# -----------------------------

plt.figure(figsize=(8,4))
plt.plot(base_late_series, label="Baseline")
plt.plot(opt_late_series, label="Optimized")
plt.xlabel("Transition Index")
plt.ylabel("Late Students")
plt.title("Lift Congestion per Transition")
plt.legend()
plt.tight_layout()
plt.savefig("lift_congestion_comparison.png")
plt.close()

optimized.to_csv("optimized_timetable.csv", index=False)


========== FINAL RESULTS ==========
Stress Test Enabled: True

Baseline Avg Distance : 54.57 m
Optimized Avg Distance: 18.08 m

Baseline Avg Late Students : 0.38
Optimized Avg Late Students: 1.13

Baseline Max Late Students : 50
Optimized Max Late Students: 50
